# RG3 gas: same-shot relationships and paper-inspired interaction
Source ideas are separated from new formulas; max/mean inconsistencies excluded.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import average_precision_score,confusion_matrix,precision_score,recall_score,f1_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
original=pd.read_csv(tables/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
data=original.loc[original['product'].eq('RG3') & (original.PassOrFail.eq(0)|original.Reason.str.contains('가스',regex=False))].reset_index(drop=True)
y=data.Reason.str.contains('가스',regex=False).astype(int)
assert len(data)==583 and y.sum()==17
p='Max_Injection_Pressure';b='Average_Back_Pressure';v='Max_Injection_Speed';s='Max_Switch_Over_Pressure';f='Filling_Time';i='Injection_Time'
X=data[[p,b,v,s,f,i]].copy()
def ratio(a,b):return a.div(b.where(b.ne(0))).replace([np.inf,-np.inf],np.nan)
X['switch_pressure_ratio']=ratio(X[s],X[p])
X['filling_time_ratio']=ratio(X[f],X[i])
X['pressure_speed_ratio']=ratio(X[p],X[v])
X['back_injection_ratio']=ratio(X[b],X[p])
# These are numerical relationships, not viscosity, energy or proven physical units.
r1,r2,r3,r4=['switch_pressure_ratio','filling_time_ratio','pressure_speed_ratio','back_injection_ratio']
interaction='standardized_pressure_interaction'
configs={
'R1_transition_time':[r1,r2],
'R2_transition_time_speed':[r1,r2,r3],
'R3_two_pressure_ratios':[r1,r4],
'R4_four_ratios':[r1,r2,r3,r4],
'P1_paper_pressure_interaction':[r1,r4,interaction],
'P2_pressure_interaction_time':[r1,r4,interaction,r2],
'raw_transition_time':[s,p,f,i],
'raw_two_pressures':[b,p,s],
'raw_pressure_speed':[p,v],
'raw_pressure_time':[b,p,f,i],
'previous_std10':['Barrel_Temperature_2__past10_std','Hopper_Temperature__past10_std','Cushion_Position__past10_std','Barrel_Temperature_5__past10_std']}
for c in configs['previous_std10']:X[c]=data[c]
groups=data.comparison_block
folds=list(LeaveOneGroupOut().split(X,y,groups))
results=[];predictions=[];blockrows=[]
for setting,cols in configs.items():
    probs=np.full(len(data),np.nan)
    for train,test in folds:
        assert set(groups.iloc[train]).isdisjoint(groups.iloc[test])
        tr=X.iloc[train].copy();te=X.iloc[test].copy()
        if interaction in cols:
            # Train-only centering/scaling prevents held-out values defining interaction.
            means=tr[[b,p]].mean();scales=tr[[b,p]].std(ddof=0).replace(0,1)
            tr[interaction]=((tr[b]-means[b])/scales[b])*((tr[p]-means[p])/scales[p])
            te[interaction]=((te[b]-means[b])/scales[b])*((te[p]-means[p])/scales[p])
        imp=SimpleImputer(strategy='median',keep_empty_features=True)
        xt=imp.fit_transform(tr[cols]);xv=imp.transform(te[cols])
        model=RandomForestClassifier(n_estimators=200,max_depth=8,min_samples_leaf=3,class_weight='balanced_subsample',random_state=42,n_jobs=-1)
        model.fit(xt,y.iloc[train]);probs[test]=model.predict_proba(xv)[:,list(model.classes_).index(1)]
    pred=probs>=.5
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    selected=np.zeros(len(data),dtype=bool)
    tie=np.random.RandomState(42).random_sample(len(data))
    for block in groups.unique():
        idx=np.flatnonzero(groups.eq(block).to_numpy())
        order=idx[np.lexsort((tie[idx],-probs[idx]))];selected[order[:int(np.ceil(len(idx)*.1))]]=True
        blockrows.append(dict(setting=setting,block=block,shots=len(idx),gas=int(y.iloc[idx].sum()),caught_top10=int((selected[idx]&y.iloc[idx].astype(bool)).sum()),TP=int((pred[idx]&y.iloc[idx].astype(bool)).sum()),FP=int((pred[idx]&~y.iloc[idx].astype(bool)).sum())))
    row=dict(setting=setting,features=len(cols),shots=len(data),gas=int(y.sum()),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),AP=average_precision_score(y,probs),F1=f1_score(y,pred,zero_division=0),precision=precision_score(y,pred,zero_division=0),recall=recall_score(y,pred,zero_division=0),caught_top10=int((selected&y.astype(bool)).sum()),selected_top10=int(selected.sum()))
    results.append(row)
    out=data[['EQUIP_CD','product','TimeStamp','comparison_block','Reason']].copy()
    out['gas']=y;out['probability']=probs;out['predicted_gas']=pred;out['selected_top10']=selected;out['setting']=setting
    predictions.append(out)
    pd.DataFrame(results).to_csv(tables/'rg3_same_shot_combo_metrics.csv',index=False,encoding='utf-8-sig')
    print(setting,'TP',tp,'FP',fp,'AP',round(row['AP'],4),'top10',row['caught_top10'],flush=True)
# Observational distribution overlap, not a fold-fitted decision rule.
distribution=[]
for block,g in data.groupby('comparison_block'):
    idx=g.index;normal=y.loc[idx].eq(0);gas=~normal
    for col in [r1,r2,r3,r4]:
        values=X.loc[idx,col];n=values[normal].dropna();bad=values[gas].dropna()
        q10,q90=n.quantile(.1),n.quantile(.9)
        distribution.append(dict(block=block,feature=col,normal_n=len(n),gas_n=len(bad),normal_median=n.median(),gas_median=bad.median(),median_difference=bad.median()-n.median(),normal_q10=q10,normal_q90=q90,gas_outside_normal_middle80=int(((bad<q10)|(bad>q90)).sum())))
metrics=pd.DataFrame(results);pred=pd.concat(predictions,ignore_index=True)
assert (metrics.TP+metrics.FN==17).all()
assert (metrics[['TP','FP','FN','TN']].sum(axis=1)==583).all()
assert len(pred)==len(configs)*583 and pred.probability.between(0,1).all()
assert not pred.duplicated(['setting','EQUIP_CD','product','TimeStamp']).any()
# Previous model reconciliation on the exact same cohort.
baseline=metrics.loc[metrics.setting.eq('previous_std10')].iloc[0]
assert baseline.TP==1 and baseline.FP==6 and abs(baseline.AP-.13871113959502832)<1e-10
pred.to_csv(tables/'rg3_same_shot_combo_oof_predictions.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(blockrows).to_csv(tables/'rg3_same_shot_combo_by_block.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(distribution).to_csv(tables/'rg3_same_shot_feature_overlap.csv',index=False,encoding='utf-8-sig')
X.join(data[['TimeStamp','comparison_block','Reason']]).assign(gas=y).to_csv(tables/'rg3_same_shot_feature_dataset.csv',index=False,encoding='utf-8-sig')
pd.DataFrame([dict(setting=k,features=len(cols),columns=';'.join(cols)) for k,cols in configs.items()]).to_csv(tables/'rg3_same_shot_combo_inputs.csv',index=False,encoding='utf-8-sig')
source='https://scienceon.kisti.re.kr/commons/util/originalView.do?cn=JAKO202519664804270&oCn=JAKO202519664804270&dbt=JAKO&journal=564564'
repo='https://github.com/Chohyungun/NOVA50101-KAMP-Injection-Molding'
lines=['# RG3 가스: 같은 Shot 변수 관계 비교','',
'정상566 Shot, 가스17 Shot. 기존과 동일한 Random Forest(200트리,깊이8,말단최소3,balanced_subsample,시드42), 관측구간 하나 전체 제외5회 평가. 확률0.5 고정, 구간별 상위약10% 총62 Shot 검사.',
'','## 출처와 새로 제안한 식','',
f'- [논문]({source}) 3.1절은 2·3차 다항식, 상호작용, 이동평균/EWMA/lag를 설명하지만 상세식·창길이는 없다. 4.3절은 평균배압–최대사출압력 상호작용을 강조한다. 본 실험의 표준화곱은 이 아이디어를 우리가 구체화한 것이며 논문의 SHAP interaction 계산을 재현한 것이 아니다.',
f'- [GitHub]({repo})의 확인한 전처리 코드와 준지도 노트북은 스케일링·불균형보정·거리/방향점수 중심이었다. 같은Shot 파생변수의 구체식은 해당검토범위에서 확인하지 못했다. 거리/방향점수는 향후 train-fold 안에서만 학습해야 한다.',
'- 아래 네 비율은 우리가 제안한 숫자 관계 후보이며 출처에서 그대로 가져온 계산식이 아니다. 압력/속도 비율을 점도라고 부르지 않는다. 센서정의와 단위 검증이 필요하다.',
'','## 데이터 정의 점검','',
f"- 최대배압<평균배압: {int(data.Max_Back_Pressure.lt(data.Average_Back_Pressure).sum())}/583 Shot. 최대RPM<평균RPM: {int(data.Max_Screw_RPM.lt(data.Average_Screw_RPM).sum())}/583 Shot. 동일단위·동일집계 정의인지 불확실해 최대−평균 차이 후보는 제외했다.",
'- Cooling_Time이 기존분석컬럼에 없어 온도–냉각시간 조합을 만들지 않았다.',
'- 비율의0분모는 결측 처리하고 학습구간 중앙값으로 대체한다. 이번사용 분모는 모두양수. filling_time_ratio는 시간컬럼의 의미와 포함관계를 확정하지 않은 숫자비율이다.',
'','## 입력','',
'| 변수 | 식 |','|---|---|',
'| 전환압력비율 | 최대전환압력 / 최대사출압력 |',
'| 충전시간비율 | 충전시간 / 사출시간 |',
'| 압력속도비율 | 최대사출압력 / 최대사출속도 |',
'| 배압사출비율 | 평균배압 / 최대사출압력 |',
'| 압력상호작용 | 학습구간에서 표준화한 평균배압 × 표준화한 최대사출압력 |',
'','| 조합 | 입력 |','|---|---|',
'| R1 | 전환압력비율 + 충전시간비율 |',
'| R2 | R1 + 압력속도비율 |',
'| R3 | 전환압력비율 + 배압사출비율 |',
'| R4 | 비율4개 |',
'| P1 | R3 + 압력상호작용 |',
'| P2 | P1 + 충전시간비율 |',
'| raw_transition_time | 최대전환압력,최대사출압력,충전시간,사출시간 |',
'| raw_two_pressures | 평균배압,최대사출압력,최대전환압력 |',
'| raw_pressure_speed | 최대사출압력,최대사출속도 |',
'| raw_pressure_time | 평균배압,최대사출압력,충전시간,사출시간 |',
'| previous_std10 | 기존배럴2·호퍼·쿠션·배럴5 이전10개 표준편차 |',
'','## 전체 결과','',
'| 조합 | 변수수 | 불량판정 | 실제가스TP | 정상오탐FP | 정상판정 | 실제정상TN | 놓친가스FN | AP | 상위10% 발견/17 |',
'|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|']
for r in metrics.itertuples():lines.append(f'| {r.setting} | {r.features} | {r.TP+r.FP} | {r.TP} | {r.FP} | {r.TN+r.FN} | {r.TN} | {r.FN} | {r.AP:.4f} | {r.caught_top10} |')
lines+=['','## 해석 범위','',
'변수선정·조합탐색에 같은평가데이터를 사용했으므로 독립최종성능이 아니다. 블록은 실제Run ID가아니고 학습에 미래블록이 들어갈수있어 엄격한미래시험과다르다. 모델성능은 가스발생원인의증거가아니다. 원값과 파생값의 비교는 입력수·표현차이까지 포함하며 개별파생변수 효과의확정시험이아니다.']
(root/'reports/rg3_same_shot_combo_comparison.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
print('Verified 11 experiments and exact prior-model reconciliation.',flush=True)



R1_transition_time TP 2 FP 86 AP 0.026 top10 2


R2_transition_time_speed TP 2 FP 55 AP 0.0285 top10 2


R3_two_pressure_ratios TP 0 FP 23 AP 0.0245 top10 1


R4_four_ratios TP 0 FP 21 AP 0.0281 top10 1


P1_paper_pressure_interaction TP 0 FP 23 AP 0.026 top10 0


...

raw_two_pressures TP 0 FP 34 AP 0.0274 top10 2


raw_pressure_speed TP 2 FP 96 AP 0.0366 top10 2


raw_pressure_time TP 2 FP 64 AP 0.0367 top10 1


previous_std10 TP 1 FP 6 AP 0.1387 top10 6


Verified 11 experiments and exact prior-model reconciliation.
